# Opportunity Layer Construction

## Overview

This notebook uses processed POI category counts derived from the Ordnance Survey Points of Interest dataset to represent the relative availability of destination categories across Greater Manchester.

The notebook applies logarithmic transformation and normalisation to construct category-level opportunity distributions for context-dependent trip purposes. These distributions are subsequently integrated with the Behaviour Layer through the trip-purpose-to-POI ontology.

## Dataset Information

| Item | Description |
|------|-------------|
| Dataset |District_manchester_poi_counts|
| Source | Ordnance Survey - Points of Interest Documentation |
| Geography | local authorities: district / unitary (as of April 2023) |
| Input File |District_manchester_poi_counts|
| Output File | |
| Notebook Author | Peter |
| Last Updated |2026-07-24|

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## 2. Set Up File Pathes

In [2]:
import sys
from pathlib import Path

# Find repository root (../..)
repo_root = Path.cwd().parents[1]

# Add repository root to Python import path
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [3]:
project_root = Path.cwd().parents[1]

input_file_path = project_root / "data" / "POI" / "processed" / "manchester" / "District_manchester_poi_counts.xlsx"

## 3. Load Processed POI Data

In [4]:
poi_counts = pd.read_excel(input_file_path)

print(f"Dataset loaded successfully.")
print(f"Number of POI categories: {len(poi_counts)}")

Dataset loaded successfully.
Number of POI categories: 10


## 4. Data Inspection

In [5]:
display(poi_counts.head(10))

,POI Category,POI Count
0,Accommodation,139
1,Education,618
2,Employment,6031
3,Food and Beverage,2731
4,Healthcare,942
5,Personal Services,1613
6,Public Services,2899
7,Recreation,1256
8,Retail,2856
9,Transport,3401


## 5. Data Processing

### 5.1 Logarithmic Transformation of POI Counts

The Opportunity Layer assumes that areas containing more POIs provide greater opportunities for activities. However, raw POI counts are highly skewed across categories, with certain categories (e.g., Employment or Retail) containing substantially more POIs than others.

To reduce the dominance of extremely large counts while preserving their relative ordering, a logarithmic transformation is applied using the natural logarithm:

\[
\log(1+x)
\]

where \(x\) is the original POI count.

The transformed values are subsequently used to construct opportunity distributions.

In [6]:
poi_counts_processed = poi_counts.copy()


def log_transform_poi_counts(poi_counts_df):
    """
    Apply log(1 + x) transformation to POI counts.
    """

    df = poi_counts_df.copy()

    df["Log Count"] = np.log1p(df["POI Count"])

    return df


poi_counts_processed = log_transform_poi_counts(poi_counts_processed)

display(poi_counts_processed)

,POI Category,POI Count,Log Count
0,Accommodation,139,4.941642
1,Education,618,6.428105
2,Employment,6031,8.704834
3,Food and Beverage,2731,7.912789
4,Healthcare,942,6.849066
5,Personal Services,1613,7.386471
6,Public Services,2899,7.972466
7,Recreation,1256,7.136483
8,Retail,2856,7.957527
9,Transport,3401,8.132119


## 6. Opportunity Distribution Construction

### 6.1 Trip Purpose-to-POI Mapping

Trip purposes generated by the Behaviour Layer are first mapped to one or more candidate POI categories.

Most trip purposes correspond directly to a single POI category (e.g., Shopping → Retail). However, certain trip purposes, such as Personal Business, may reasonably occur across multiple POI categories.

The mapping below defines the candidate POI categories used when constructing opportunity distributions.

In [7]:
from ontology.TRIP_PURPOSE_TO_POI_MAPPING import TRIP_PURPOSE_TO_POI_MAPPING

TRIP_PURPOSE_TO_POI_MAPPING

,Unified Trip Purpose,Mapping Strategy,POI Categories
0,Commuting,Representative Mapping,[Employment]
1,Business,Representative Mapping,[Employment]
2,Visiting Friends Elsewhere,Representative Mapping,[Food and Beverage]
3,Entertainment or Public Activity,Representative Mapping,[Recreation]
4,Day Trip,Representative Mapping,[Recreation]
5,Just Walk,Representative Mapping,[Recreation]
6,Education,Direct Mapping,[Education]
7,Escort Education,Direct Mapping,[Education]
8,Shopping,Direct Mapping,[Retail]
9,Sport: Participate,Direct Mapping,[Recreation]


### 6.2 Construction of Opportunity Distributions

For Direct and Representative mappings, the mapped POI category is assigned an opportunity weight of 1.

For Context-dependent mappings, the log-transformed POI counts are normalised within the candidate category set of each trip purpose:

\[
w_{p,c} =
\frac{\log(1+n_c)}
{\sum_{j \in C_p}\log(1+n_j)}
\]

where \(n_c\) is the number of POIs in category \(c\), \(C_p\) is the candidate category set for trip purpose \(p\), and \(w_{p,c}\) is the resulting opportunity weight.

Trip purposes classified as Excluded Mapping are not assigned a POI-based opportunity distribution.

In [8]:
def build_opportunity_distribution(mapping_df, poi_counts_df):
    """
    Construct category-level opportunity distributions for trip purposes.

    Direct and Representative mappings receive a weight of 1 for their
    single mapped POI category. Context-dependent mappings are weighted
    using normalised log-transformed POI counts. Excluded mappings are
    omitted from the output.

    Parameters
    ----------
    mapping_df : pandas.DataFrame
        Trip-purpose-to-POI mapping ontology.

    poi_counts_df : pandas.DataFrame
        POI category counts containing a precomputed 'Log Count' column.

    Returns
    -------
    pandas.DataFrame
        Long-format opportunity distribution by trip purpose and
        POI category.
    """

    required_mapping_columns = {
        "Unified Trip Purpose",
        "Mapping Strategy",
        "POI Categories"
    }

    required_poi_columns = {
        "POI Category",
        "Log Count"
    }

    # Check that all required columns are available
    missing_mapping_columns = (
        required_mapping_columns - set(mapping_df.columns)
    )

    missing_poi_columns = (
        required_poi_columns - set(poi_counts_df.columns)
    )

    if missing_mapping_columns:
        raise ValueError(
            f"Missing mapping columns: "
            f"{sorted(missing_mapping_columns)}"
        )

    if missing_poi_columns:
        raise ValueError(
            f"Missing POI columns: "
            f"{sorted(missing_poi_columns)}"
        )

    # Each POI category should have only one count
    if poi_counts_df["POI Category"].duplicated().any():
        raise ValueError(
            "POI Category must be unique in poi_counts_df."
        )

    # Create a lookup from POI category to log-transformed count
    log_count_lookup = (
        poi_counts_df
        .set_index("POI Category")["Log Count"]
    )

    output_rows = []

    # Process each trip-purpose mapping
    for _, mapping_row in mapping_df.iterrows():

        trip_purpose = mapping_row["Unified Trip Purpose"]
        strategy = mapping_row["Mapping Strategy"]
        categories = mapping_row["POI Categories"]

        # Excluded purposes do not receive a POI distribution
        if strategy == "Excluded Mapping":
            continue

        if not isinstance(categories, (list, tuple)):
            raise TypeError(
                f"POI Categories for '{trip_purpose}' "
                f"must be a list or tuple."
            )

        if not categories:
            raise ValueError(
                f"No POI categories were defined for "
                f"'{trip_purpose}'."
            )

        if len(categories) != len(set(categories)):
            raise ValueError(
                f"Duplicate POI categories were defined for "
                f"'{trip_purpose}'."
            )

        # Check that every mapped category has a POI count
        missing_categories = [
            category
            for category in categories
            if category not in log_count_lookup.index
        ]

        if missing_categories:
            raise ValueError(
                f"Missing POI counts for '{trip_purpose}': "
                f"{missing_categories}"
            )

        # Direct and Representative mappings use one category
        if strategy in {
            "Direct Mapping",
            "Representative Mapping"
        }:

            if len(categories) != 1:
                raise ValueError(
                    f"'{trip_purpose}' uses {strategy} but has "
                    f"{len(categories)} POI categories."
                )

            weights = [1.0]

        # Context-dependent mappings use normalised log counts
        elif strategy == "Context-dependent Mapping":

            candidate_log_counts = (
                log_count_lookup.loc[list(categories)]
            )

            total_log_count = candidate_log_counts.sum()

            if total_log_count <= 0:
                raise ValueError(
                    f"Total Log Count must be positive for "
                    f"'{trip_purpose}'."
                )

            weights = (
                candidate_log_counts / total_log_count
            ).tolist()

        else:
            raise ValueError(
                f"Unknown mapping strategy for "
                f"'{trip_purpose}': {strategy}"
            )

        # Convert the result into long format
        for category, weight in zip(categories, weights):
            output_rows.append({
                "Unified Trip Purpose": trip_purpose,
                "Mapping Strategy": strategy,
                "POI Category": category,
                "Opportunity Weight": weight
            })

    return pd.DataFrame(
        output_rows,
        columns=[
            "Unified Trip Purpose",
            "Mapping Strategy",
            "POI Category",
            "Opportunity Weight"
        ]
    )

The opportunity distribution is constructed by applying the mapping rules to each unified trip purpose.

In [9]:
opportunity_distribution = build_opportunity_distribution(
    mapping_df=TRIP_PURPOSE_TO_POI_MAPPING,
    poi_counts_df=poi_counts_processed
)

display(
    opportunity_distribution.style.format({
        "Opportunity Weight": "{:.4f}"
    })
)

,Unified Trip Purpose,Mapping Strategy,POI Category,Opportunity Weight
0,Commuting,Representative Mapping,Employment,1.0000
1,Business,Representative Mapping,Employment,1.0000
2,Visiting Friends Elsewhere,Representative Mapping,Food and Beverage,1.0000
3,Entertainment or Public Activity,Representative Mapping,Recreation,1.0000
4,Day Trip,Representative Mapping,Recreation,1.0000
5,Just Walk,Representative Mapping,Recreation,1.0000
6,Education,Direct Mapping,Education,1.0000
7,Escort Education,Direct Mapping,Education,1.0000
8,Shopping,Direct Mapping,Retail,1.0000
9,Sport: Participate,Direct Mapping,Recreation,1.0000


### 6.3 Validation

The resulting distributions are validated to ensure that all weights are within the interval \([0,1]\), sum to 1 within each trip purpose, and contain no duplicated trip-purpose-category combinations.

In [10]:
# All weights must be between 0 and 1
assert opportunity_distribution[
    "Opportunity Weight"
].between(0, 1).all()


# Weights must sum to 1 within each trip purpose
weight_sums = (
    opportunity_distribution
    .groupby("Unified Trip Purpose")["Opportunity Weight"]
    .sum()
)

assert np.allclose(
    weight_sums.to_numpy(),
    1.0
)


# Each trip-purpose-category combination must be unique
assert not opportunity_distribution.duplicated(
    subset=[
        "Unified Trip Purpose",
        "POI Category"
    ]
).any()


# Excluded mappings must not appear in the output
assert (
    opportunity_distribution["Mapping Strategy"]
    != "Excluded Mapping"
).all()


print("All opportunity distribution validation checks passed.")

All opportunity distribution validation checks passed.
